# COMP6002 — Federated Prototype V1

This notebook builds the first working prototype for the diabetes project.

**Design:** each dataset trains its own full-feature local teacher model. Because the datasets have different feature spaces, the teachers are not averaged directly. Their knowledge is distilled locally into identical student networks built on a small shared semantic feature space (`age`, `bmi`, `hba1c`, `hypertension`). Those student networks can then be combined using weighted FedAvg.

This is a research prototype, not a clinical diagnostic system.

In [1]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)
print("PyTorch:", torch.__version__)

Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.2.2


## 1. Load datasets

In [2]:
DATASET1_PATH = "../data/diabetes_prediction_dataset.csv"
DATASET2_PATH = "../data/diabetes_dataset.csv"

df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)

print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)
display(df1.head())
display(df2.head())

Dataset 1: (100000, 9)
Dataset 2: (100000, 31)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,cardiovascular_history,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,7.9,7.9,0,0,0,30.5,0.89,134,78,68,239,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,6.5,8.7,0,0,0,23.1,0.80,129,76,67,116,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,10.0,8.1,1,0,0,22.2,0.81,115,73,74,213,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,6.6,5.2,0,0,0,26.8,0.88,120,93,68,171,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,7.4,5.0,0,0,0,21.2,0.78,92,67,67,210,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


## 2. Targets and leakage removal

Dataset 2 excludes `diabetes_stage` because it directly reveals disease status. `diabetes_risk_score` is also excluded from Prototype V1 because its derivation may encode target-related information.

In [3]:
TARGET1 = "diabetes"
TARGET2 = "diagnosed_diabetes"

df2_model = df2.drop(columns=[c for c in ["diabetes_stage", "diabetes_risk_score"] if c in df2.columns]).copy()

X1 = df1.drop(columns=[TARGET1])
y1 = df1[TARGET1].astype(int)

X2 = df2_model.drop(columns=[TARGET2])
y2 = df2_model[TARGET2].astype(int)

print("Dataset 1 target:")
print(y1.value_counts(normalize=True).sort_index().round(4))
print("\nDataset 2 target:")
print(y2.value_counts(normalize=True).sort_index().round(4))

Dataset 1 target:
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64

Dataset 2 target:
diagnosed_diabetes
0    0.4
1    0.6
Name: proportion, dtype: float64


In [4]:
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=SEED, stratify=y1
)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=SEED, stratify=y2
)

print("Dataset 1 split:", X1_train.shape, X1_test.shape)
print("Dataset 2 split:", X2_train.shape, X2_test.shape)

Dataset 1 split: (80000, 8) (20000, 8)
Dataset 2 split: (80000, 28) (20000, 28)


## 3. Local teacher preprocessing and training

In [5]:
def build_preprocessor(X):
    categorical = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = [c for c in X.columns if c not in categorical]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numerical),
        ("cat", categorical_pipe, categorical)
    ])

teacher1 = Pipeline([
    ("preprocess", build_preprocessor(X1_train)),
    ("model", RandomForestClassifier(
        n_estimators=250, random_state=SEED,
        class_weight="balanced", n_jobs=-1
    ))
])

teacher2 = Pipeline([
    ("preprocess", build_preprocessor(X2_train)),
    ("model", RandomForestClassifier(
        n_estimators=250, random_state=SEED,
        class_weight="balanced", n_jobs=-1
    ))
])

print("Training Teacher A...")
teacher1.fit(X1_train, y1_train)
print("Training Teacher B...")
teacher2.fit(X2_train, y2_train)
print("Done.")

Training Teacher A...
Training Teacher B...
Done.


In [6]:
def evaluate_classifier(name, y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }
    print("\n", name)
    print(pd.Series(row).drop("Model"))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, pred))
    return row

results = []
p1 = teacher1.predict_proba(X1_test)[:, 1]
p2 = teacher2.predict_proba(X2_test)[:, 1]

results.append(evaluate_classifier("Teacher A - Dataset 1", y1_test, p1))
results.append(evaluate_classifier("Teacher B - Dataset 2", y2_test, p2))


 Teacher A - Dataset 1
Accuracy      0.96935
Precision    0.931692
Recall           0.69
F1           0.792835
ROC-AUC      0.966138
dtype: object
Confusion matrix:
[[18214    86]
 [  527  1173]]

 Teacher B - Dataset 2
Accuracy       0.9197
Precision    0.999423
Recall       0.866667
F1           0.928323
ROC-AUC      0.940935
dtype: object
Confusion matrix:
[[ 7994     6]
 [ 1600 10400]]


## 4. Shared semantic view

The local teachers still use all suitable local features. The student models use only four concepts that can be reasonably mapped across both datasets:

- age
- BMI
- HbA1c
- hypertension

This shared view allows both students to have the same input dimension and architecture.

In [7]:
def shared_view_dataset1(X):
    return pd.DataFrame({
        "age": pd.to_numeric(X["age"], errors="coerce"),
        "bmi": pd.to_numeric(X["bmi"], errors="coerce"),
        "hba1c": pd.to_numeric(X["HbA1c_level"], errors="coerce"),
        "hypertension": pd.to_numeric(X["hypertension"], errors="coerce")
    }, index=X.index)

def shared_view_dataset2(X):
    return pd.DataFrame({
        "age": pd.to_numeric(X["age"], errors="coerce"),
        "bmi": pd.to_numeric(X["bmi"], errors="coerce"),
        "hba1c": pd.to_numeric(X["hba1c"], errors="coerce"),
        "hypertension": pd.to_numeric(X["hypertension_history"], errors="coerce")
    }, index=X.index)

S1_train = shared_view_dataset1(X1_train)
S1_test = shared_view_dataset1(X1_test)
S2_train = shared_view_dataset2(X2_train)
S2_test = shared_view_dataset2(X2_test)

display(S1_train.head())
display(S2_train.head())

,age,bmi,hba1c,hypertension
74736,80.0,27.32,6.5,1
36589,19.0,25.18,4.5,0
37414,36.0,25.95,6.6,0
71251,35.0,23.43,6.0,0
40454,30.0,22.62,5.0,0


,age,bmi,hba1c,hypertension
27589,21,26.0,6.54,0
84374,66,26.9,6.15,0
69440,30,15.0,7.95,0
66183,42,27.1,5.92,0
16613,53,24.6,8.10,1


In [8]:
# Local scaling for the prototype
scaler1 = StandardScaler()
scaler2 = StandardScaler()

S1_train_scaled = scaler1.fit_transform(S1_train)
S1_test_scaled = scaler1.transform(S1_test)
S2_train_scaled = scaler2.fit_transform(S2_train)
S2_test_scaled = scaler2.transform(S2_test)

# Teacher probability targets used for local knowledge distillation
teacher1_train_soft = teacher1.predict_proba(X1_train)[:, 1]
teacher2_train_soft = teacher2.predict_proba(X2_train)[:, 1]

## 5. Student network and local distillation

In [9]:
class StudentNet(nn.Module):
    def __init__(self, input_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def train_student(X, soft_targets, epochs=20, batch_size=512, lr=1e-3):
    model = StudentNet(input_dim=X.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(soft_targets, dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | loss={total_loss / len(X_t):.5f}")
    return model

print("Student A")
student1 = train_student(S1_train_scaled, teacher1_train_soft)

print("\nStudent B")
student2 = train_student(S2_train_scaled, teacher2_train_soft)

Student A
Epoch 05 | loss=0.17905
Epoch 10 | loss=0.17428
Epoch 15 | loss=0.16962
Epoch 20 | loss=0.16569

Student B
Epoch 05 | loss=0.27858
Epoch 10 | loss=0.27255
Epoch 15 | loss=0.26970
Epoch 20 | loss=0.26533


## 6. Federated aggregation

Because Student A and Student B have identical architectures, their parameters can be combined using dataset-size-weighted FedAvg.

In [10]:
def fedavg(models, sample_counts):
    total = float(sum(sample_counts))
    global_state = copy.deepcopy(models[0].state_dict())

    for key in global_state:
        global_state[key] = torch.zeros_like(global_state[key])
        for model, n in zip(models, sample_counts):
            global_state[key] += model.state_dict()[key] * (n / total)

    global_model = StudentNet(input_dim=4)
    global_model.load_state_dict(global_state)
    return global_model

global_student = fedavg(
    [student1, student2],
    [len(S1_train_scaled), len(S2_train_scaled)]
)

print("Federated global student created.")

Federated global student created.


In [11]:
def predict_student(model, X):
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        return torch.sigmoid(model(X_t)).cpu().numpy()

local1_prob = predict_student(student1, S1_test_scaled)
local2_prob = predict_student(student2, S2_test_scaled)
fed1_prob = predict_student(global_student, S1_test_scaled)
fed2_prob = predict_student(global_student, S2_test_scaled)

results.append(evaluate_classifier("Local Student A - Dataset 1", y1_test, local1_prob))
results.append(evaluate_classifier("Local Student B - Dataset 2", y2_test, local2_prob))
results.append(evaluate_classifier("Federated Student - Dataset 1", y1_test, fed1_prob))
results.append(evaluate_classifier("Federated Student - Dataset 2", y2_test, fed2_prob))

results_df = pd.DataFrame(results)
for c in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    results_df[c] = results_df[c].round(4)

results_df


 Local Student A - Dataset 1
Accuracy      0.94345
Precision     0.92911
Recall       0.362353
F1           0.521371
ROC-AUC        0.9341
dtype: object
Confusion matrix:
[[18253    47]
 [ 1084   616]]

 Local Student B - Dataset 2
Accuracy       0.9041
Precision    0.982854
Recall       0.855083
F1           0.914528
ROC-AUC        0.9349
dtype: object
Confusion matrix:
[[ 7821   179]
 [ 1739 10261]]

 Federated Student - Dataset 1
Accuracy      0.61115
Precision    0.175547
Recall       0.967059
F1           0.297153
ROC-AUC      0.912625
dtype: object
Confusion matrix:
[[10579  7721]
 [   56  1644]]

 Federated Student - Dataset 2
Accuracy       0.7732
Precision    0.972764
Recall       0.639917
F1           0.771992
ROC-AUC      0.907299
dtype: object
Confusion matrix:
[[7785  215]
 [4321 7679]]


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Teacher A - Dataset 1,0.9694,0.9317,0.6900,0.7928,0.9661
1,Teacher B - Dataset 2,0.9197,0.9994,0.8667,0.9283,0.9409
2,Local Student A - Dataset 1,0.9434,0.9291,0.3624,0.5214,0.9341
3,Local Student B - Dataset 2,0.9041,0.9829,0.8551,0.9145,0.9349
4,Federated Student - Dataset 1,0.6112,0.1755,0.9671,0.2972,0.9126
5,Federated Student - Dataset 2,0.7732,0.9728,0.6399,0.7720,0.9073


## 7. Save prototype outputs

In [12]:
os.makedirs("../models", exist_ok=True)
torch.save(global_student.state_dict(), "../models/federated_student_v1.pt")
results_df.to_csv("../models/prototype_v1_metrics.csv", index=False)

print("Saved ../models/federated_student_v1.pt")
print("Saved ../models/prototype_v1_metrics.csv")

Saved ../models/federated_student_v1.pt
Saved ../models/prototype_v1_metrics.csv
